In [1]:


import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sksurv.metrics import concordance_index_censored
import pickle
import json
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
print("All imports successful")

Using device: cpu
PyTorch version: 2.12.0
All imports successful


In [3]:
import os
# If running on Colab, mount Google Drive or upload files
# Adjust paths to wherever your files are in Colab
os.chdir('/Users/parthshringarpure/Desktop/AI/Projects/luad_survival')
# Load all three feature streams
expr    = pd.read_csv('data/processed/expression_matrix.csv', index_col=0)      # 478 × 1000
dysreg  = pd.read_csv('data/processed/dysregulation_scores.csv', index_col=0)   # 478 × 819
immune  = pd.read_csv('data/processed/immune_features_cibersort.csv', index_col=0)  # 478 × 22
clinical = pd.read_csv('data/processed/clinical_survival.csv', index_col=0)     # 478 × 10

# Align all patients — intersection of all four
common = expr.index.intersection(dysreg.index).intersection(immune.index).intersection(clinical.index)
expr     = expr.loc[common]
dysreg   = dysreg.loc[common]
immune   = immune.loc[common]
clinical = clinical.loc[common]

print(f"Patients aligned: {len(common)}")
print(f"Expression:       {expr.shape}")
print(f"Dysregulation:    {dysreg.shape}")
print(f"Immune:           {immune.shape}")
print(f"Clinical:         {clinical.shape}")

Patients aligned: 478
Expression:       (478, 1000)
Dysregulation:    (478, 819)
Immune:           (478, 22)
Clinical:         (478, 10)


In [38]:
# Build structured survival array
y = np.array(
    [(bool(e), t) for e, t in zip(clinical['event'], clinical['survival_time'])],
    dtype=[('event', bool), ('time', float)]
)

print(f"Total patients:  {len(y)}")
print(f"Events (deaths): {y['event'].sum()}  ({y['event'].mean()*100:.1f}%)")
print(f"Censored:        {(~y['event']).sum()}  ({(~y['event']).mean()*100:.1f}%)")
print(f"Survival time:   {y['time'].min():.0f} to {y['time'].max():.0f} days")

Total patients:  478
Events (deaths): 121  (25.3%)
Censored:        357  (74.7%)
Survival time:   1 to 6812 days


In [39]:
class Encoder(nn.Module):
    """
    Compresses one input stream down to 32 dimensions.
    Same structure for all three streams — only input_dim differs.
    """
    def __init__(self, input_dim, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32)
        )

    def forward(self, x):
        return self.net(x)


class FusionModel(nn.Module):
    """
    Three stream fusion model with attention.

    Streams:
      - Expression:    1000 genes  → Encoder → 32 dims
      - Dysregulation: 819 genes   → Encoder → 32 dims
      - Immune:        22 features → Encoder → 32 dims

    Attention:
      - Stack 3 × 32 summaries → (batch, 3, 32)
      - Learn 3 scalar weights per patient via small network
      - Softmax so weights sum to 1.0
      - Weighted sum → 32 dims

    Output:
      - Linear(32 → 1) → risk score
    """
    def __init__(self, expr_dim=1000, dysreg_dim=819, immune_dim=22, dropout=0.3):
        super().__init__()

        # Three encoders — one per stream
        self.encoder_expr   = Encoder(expr_dim, dropout)
        self.encoder_dysreg = Encoder(dysreg_dim, dropout)
        self.encoder_immune = Encoder(immune_dim, dropout)

        # Attention network — takes 32-dim summary, outputs 1 scalar weight
        self.attention = nn.Sequential(
            nn.Linear(32, 16),
            nn.Tanh(),
            nn.Linear(16, 1)
        )

        # Final risk score output
        self.output = nn.Linear(32, 1)

    def forward(self, x_expr, x_dysreg, x_immune):
        # Encode each stream → 32 dims
        h_expr   = self.encoder_expr(x_expr)      # (batch, 32)
        h_dysreg = self.encoder_dysreg(x_dysreg)  # (batch, 32)
        h_immune = self.encoder_immune(x_immune)  # (batch, 32)

        # Stack → (batch, 3, 32)
        streams = torch.stack([h_expr, h_dysreg, h_immune], dim=1)

        # Attention weights — one scalar per stream per patient
        attn_weights = self.attention(streams)         # (batch, 3, 1)
        attn_weights = torch.softmax(attn_weights, dim=1)  # sum to 1 across 3 streams

        # Weighted sum → (batch, 32)
        fused = (attn_weights * streams).sum(dim=1)

        # Risk score → (batch, 1)
        risk = self.output(fused)

        return risk, attn_weights.squeeze(-1)  # return weights for analysis


# Quick architecture test
model_test = FusionModel().to(device)
x_expr_test   = torch.randn(4, 1000).to(device)
x_dysreg_test = torch.randn(4, 819).to(device)
x_immune_test = torch.randn(4, 22).to(device)

risk_test, attn_test = model_test(x_expr_test, x_dysreg_test, x_immune_test)
print(f"Architecture test passed!")
print(f"  Input:  expr(4,1000) + dysreg(4,819) + immune(4,22)")
print(f"  Risk output shape:      {risk_test.shape}   (batch × 1)")
print(f"  Attention weights shape: {attn_test.shape}  (batch × 3 streams)")
print(f"  Attention weights sum:   {attn_test.sum(dim=1)}  (should be ~1.0 per patient)")

Architecture test passed!
  Input:  expr(4,1000) + dysreg(4,819) + immune(4,22)
  Risk output shape:      torch.Size([4, 1])   (batch × 1)
  Attention weights shape: torch.Size([4, 3])  (batch × 3 streams)
  Attention weights sum:   tensor([1., 1., 1., 1.], device='cuda:0', grad_fn=<SumBackward1>)  (should be ~1.0 per patient)


In [40]:
def cox_loss(risk_scores, times, events):
    """
    Cox partial likelihood loss.

    For each patient who died, we compare their risk score
    against all patients who were still alive at that time.
    Higher risk score for the patient who died = good prediction.

    This is the same loss DeepSurv uses.
    """
    # Sort by survival time descending
    order = torch.argsort(times, descending=True)
    risk_scores = risk_scores[order].squeeze()
    events = events[order]

    # Log-sum-exp of risk scores for patients at risk
    log_cumsum = torch.logcumsumexp(risk_scores, dim=0)

    # Loss = negative partial log likelihood for events only
    loss = -torch.mean((risk_scores - log_cumsum)[events.bool()])
    return loss


class SurvivalDataset(Dataset):
    """
    PyTorch Dataset for our three-stream fusion model.
    Returns expression, dysregulation, immune, time, event per patient.
    """
    def __init__(self, expr, dysreg, immune, times, events):
        self.expr   = torch.FloatTensor(expr)
        self.dysreg = torch.FloatTensor(dysreg)
        self.immune = torch.FloatTensor(immune)
        self.times  = torch.FloatTensor(times)
        self.events = torch.FloatTensor(events)

    def __len__(self):
        return len(self.times)

    def __getitem__(self, idx):
        return (self.expr[idx], self.dysreg[idx], self.immune[idx],
                self.times[idx], self.events[idx])


print("Cox loss function defined ✅")
print("SurvivalDataset defined ✅")

# Quick loss test
risk_test  = torch.randn(10, 1).to(device)
times_test = torch.randint(1, 1000, (10,)).float().to(device)
events_test = torch.randint(0, 2, (10,)).float().to(device)
loss_test = cox_loss(risk_test, times_test, events_test)
print(f"Loss test: {loss_test.item():.4f} ✅")

Cox loss function defined ✅
SurvivalDataset defined ✅
Loss test: 1.4627 ✅


In [41]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

fold_cindex       = []
fold_attn_weights = []

print("Running leakage-free 5-fold CV on fusion model...")
print(f"{'Fold':<6} {'Best Epoch':<12} {'Test C-index':<12}")
print("-" * 32)

for fold, (train_idx, test_idx) in enumerate(kf.split(expr), 1):

    # Split all three streams
    expr_train,   expr_test   = expr.iloc[train_idx],   expr.iloc[test_idx]
    dysreg_train, dysreg_test = dysreg.iloc[train_idx], dysreg.iloc[test_idx]
    immune_train, immune_test = immune.iloc[train_idx], immune.iloc[test_idx]
    y_train,      y_test      = y[train_idx],           y[test_idx]

    times_train  = y_train['time'].copy()
    events_train = y_train['event'].copy()
    times_test   = y_test['time'].copy()
    events_test  = y_test['event'].copy()

    # Scale each stream on training data only
    scaler_expr   = StandardScaler()
    scaler_dysreg = StandardScaler()
    scaler_immune = StandardScaler()

    expr_train_s   = scaler_expr.fit_transform(expr_train)
    expr_test_s    = scaler_expr.transform(expr_test)

    dysreg_train_s = scaler_dysreg.fit_transform(dysreg_train)
    dysreg_test_s  = scaler_dysreg.transform(dysreg_test)

    immune_train_s = scaler_immune.fit_transform(immune_train)
    immune_test_s  = scaler_immune.transform(immune_test)

    # Build dataset and dataloader
    train_dataset = SurvivalDataset(
        expr_train_s, dysreg_train_s, immune_train_s,
        times_train.copy(), events_train.copy())
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

    # Validation tensors (20% of train)
    val_size   = int(0.2 * len(train_idx))
    val_expr   = torch.FloatTensor(expr_train_s[:val_size])
    val_dysreg = torch.FloatTensor(dysreg_train_s[:val_size])
    val_immune = torch.FloatTensor(immune_train_s[:val_size])
    val_times  = times_train[:val_size].copy()
    val_events = events_train[:val_size].copy()

    # Train model
    model = FusionModel(
        expr_dim=1000, dysreg_dim=819, immune_dim=22
    ).to(device)

    model, best_val_ci, best_epoch = train_fusion_model(
        model, train_loader,
        val_expr, val_dysreg, val_immune,
        val_times, val_events,
        epochs=200, patience=20, lr=0.001
    )

    # Evaluate on test fold
    model.eval()
    with torch.no_grad():
        test_risk, test_attn = model(
            torch.FloatTensor(expr_test_s).to(device),
            torch.FloatTensor(dysreg_test_s).to(device),
            torch.FloatTensor(immune_test_s).to(device)
        )

    test_risk = test_risk.squeeze().cpu().numpy()
    test_attn = test_attn.cpu().numpy()

    ci_test = concordance_index_censored(
        events_test.astype(bool), times_test, test_risk)[0]

    fold_cindex.append(ci_test)
    fold_attn_weights.append(test_attn)

    print(f"{fold:<6} {best_epoch:<12} {ci_test:.4f}")

print("-" * 32)
print(f"\nFusion Model C-index: {np.mean(fold_cindex):.3f} ± {np.std(fold_cindex):.3f}")
print(f"\nModel comparison:")
print(f"  Cox Clinical:          0.700")
print(f"  Cox-Lasso Expression:  0.649")
print(f"  Cox-Lasso Dysreg:      0.559")
print(f"  Cox-Lasso Immune:      0.541")
print(f"  DeepSurv Expression:   0.569")
print(f"  Fusion Model:          {np.mean(fold_cindex):.3f}")

Running leakage-free 5-fold CV on fusion model...
Fold   Best Epoch   Test C-index
--------------------------------
1      87           0.5634
2      49           0.6868
3      71           0.6032
4      68           0.5892
5      53           0.6557
--------------------------------

Fusion Model C-index: 0.620 ± 0.045

Model comparison:
  Cox Clinical:          0.700
  Cox-Lasso Expression:  0.649
  Cox-Lasso Dysreg:      0.559
  Cox-Lasso Immune:      0.541
  DeepSurv Expression:   0.569
  Fusion Model:          0.620


In [42]:
# Extract and encode clinical features
# Same 5 features used in Notebook 01 Cox model

# Age — continuous, already numeric
age = clinical[['age']].copy()

# Gender — binary encode (female=0, male=1)
gender = (clinical['gender'] == 'MALE').astype(int).to_frame()

# Stage — dummy encode (Stage I as reference)
stage_dummies = pd.get_dummies(clinical['stage_group'], prefix='stage')
stage_dummies = stage_dummies.drop(columns=['stage_Stage I'], errors='ignore')

# Combine into clinical feature matrix
clinical_features = pd.concat([age, gender, stage_dummies], axis=1)
clinical_features = clinical_features.fillna(0)

print(f"Clinical features shape: {clinical_features.shape}")
print(f"Feature names: {list(clinical_features.columns)}")
print(f"\nFirst 3 patients:")
print(clinical_features.head(3))

Clinical features shape: (478, 5)
Feature names: ['age', 'gender', 'stage_Stage II', 'stage_Stage III', 'stage_Stage IV']

First 3 patients:
               age  gender  stage_Stage II  stage_Stage III  stage_Stage IV
tcga-05-4249  67.0       0           False            False           False
tcga-05-4250  79.0       0           False             True           False
tcga-05-4382  68.0       0           False            False           False


In [43]:
print(clinical['gender'].value_counts())
print(clinical['gender'].unique())

gender
female    257
male      221
Name: count, dtype: int64
['male' 'female']


In [44]:
# Fix gender encoding
gender = (clinical['gender'] == 'male').astype(float).to_frame()

# Rebuild clinical features with fixed gender
clinical_features = pd.concat([age, gender, stage_dummies], axis=1)
clinical_features = clinical_features.astype(float)
clinical_features = clinical_features.fillna(0)

print(f"Gender value counts:")
print(clinical_features['gender'].value_counts())
print(f"\nValue ranges:")
for col in clinical_features.columns:
    print(f"  {col:<20} min={clinical_features[col].min():.1f}  max={clinical_features[col].max():.1f}")
print(f"\nAny NaN: {clinical_features.isna().any().any()}")

Gender value counts:
gender
0.0    257
1.0    221
Name: count, dtype: int64

Value ranges:
  age                  min=38.0  max=88.0
  gender               min=0.0  max=1.0
  stage_Stage II       min=0.0  max=1.0
  stage_Stage III      min=0.0  max=1.0
  stage_Stage IV       min=0.0  max=1.0

Any NaN: False


In [45]:
class FusionModelV3(nn.Module):
    """
    Simplified fusion model — right-sized for 478 patients.

    Key changes vs V2:
    - Smaller encoders (less overfitting)
    - Higher dropout (0.5)
    - Expression input reduced to top genes selected per fold
    """
    def __init__(self, expr_dim=100, dysreg_dim=50,
                 immune_dim=22, clinical_dim=5, dropout=0.5):
        super().__init__()

        # Simpler encoders — one hidden layer each
        self.encoder_expr = nn.Sequential(
            nn.Linear(expr_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32)
        )

        self.encoder_dysreg = nn.Sequential(
            nn.Linear(dysreg_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32)
        )

        self.encoder_immune = nn.Sequential(
            nn.Linear(immune_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 32)
        )

        self.encoder_clinical = nn.Sequential(
            nn.Linear(clinical_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 32)
        )

        self.attention = nn.Sequential(
            nn.Linear(32, 16),
            nn.Tanh(),
            nn.Linear(16, 1)
        )

        self.output = nn.Linear(32, 1)

    def forward(self, x_expr, x_dysreg, x_immune, x_clinical):
        h_expr     = self.encoder_expr(x_expr)
        h_dysreg   = self.encoder_dysreg(x_dysreg)
        h_immune   = self.encoder_immune(x_immune)
        h_clinical = self.encoder_clinical(x_clinical)

        streams      = torch.stack([h_expr, h_dysreg, h_immune, h_clinical], dim=1)
        attn_weights = torch.softmax(self.attention(streams), dim=1)
        fused        = (attn_weights * streams).sum(dim=1)

        return self.output(fused), attn_weights.squeeze(-1)


# Test
model_test = FusionModelV3().to(device)
r, a = model_test(
    torch.randn(4, 100).to(device),
    torch.randn(4, 50).to(device),
    torch.randn(4, 22).to(device),
    torch.randn(4, 5).to(device)
)
print(f"V3 test passed — risk: {r.shape}, attn: {a.shape}")

V3 test passed — risk: torch.Size([4, 1]), attn: torch.Size([4, 4])


In [47]:
from sklearn.model_selection import StratifiedKFold

# Stratify by event so each fold has ~25% deaths
# This reduces the variance we saw (±0.077) across folds
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_cindex       = []
fold_attn_weights = []

print("Running leakage-free 5-fold CV — FusionModelV3...")
print("Gene selection inside each fold (top 100 expr, top 50 dysreg)")
print(f"{'Fold':<6} {'Best Epoch':<12} {'Test C-index':<12}")
print("-" * 32)

for fold, (train_idx, test_idx) in enumerate(kf.split(expr, y['event']), 1):
    expr_train,     expr_test     = expr.iloc[train_idx],              expr.iloc[test_idx]
    dysreg_train,   dysreg_test   = dysreg.iloc[train_idx],            dysreg.iloc[test_idx]
    immune_train,   immune_test   = immune.iloc[train_idx],            immune.iloc[test_idx]
    clinical_train, clinical_test = clinical_features.iloc[train_idx], clinical_features.iloc[test_idx]
    y_train,        y_test        = y[train_idx],                      y[test_idx]

    times_train  = y_train['time'].copy()
    events_train = y_train['event'].copy()
    times_test   = y_test['time'].copy()
    events_test  = y_test['event'].copy()

    # Gene selection on training patients only
    top_expr_genes   = expr_train.var(axis=0).nlargest(100).index
    top_dysreg_genes = dysreg_train.var(axis=0).nlargest(50).index

    expr_train_sel   = expr_train[top_expr_genes]
    expr_test_sel    = expr_test[top_expr_genes]
    dysreg_train_sel = dysreg_train[top_dysreg_genes]
    dysreg_test_sel  = dysreg_test[top_dysreg_genes]

    # Scale on training only
    scaler_expr     = StandardScaler()
    scaler_dysreg   = StandardScaler()
    scaler_immune   = StandardScaler()
    scaler_clinical = StandardScaler()

    expr_train_s     = scaler_expr.fit_transform(expr_train_sel)
    expr_test_s      = scaler_expr.transform(expr_test_sel)
    dysreg_train_s   = scaler_dysreg.fit_transform(dysreg_train_sel)
    dysreg_test_s    = scaler_dysreg.transform(dysreg_test_sel)
    immune_train_s   = scaler_immune.fit_transform(immune_train)
    immune_test_s    = scaler_immune.transform(immune_test)
    clinical_train_s = scaler_clinical.fit_transform(clinical_train)
    clinical_test_s  = scaler_clinical.transform(clinical_test)

    val_size     = int(0.2 * len(train_idx))
    val_expr     = torch.FloatTensor(expr_train_s[:val_size])
    val_dysreg   = torch.FloatTensor(dysreg_train_s[:val_size])
    val_immune   = torch.FloatTensor(immune_train_s[:val_size])
    val_clinical = torch.FloatTensor(clinical_train_s[:val_size])
    val_times    = times_train[:val_size].copy()
    val_events   = events_train[:val_size].copy()

    train_dataset = SurvivalDatasetV2(
        expr_train_s, dysreg_train_s, immune_train_s, clinical_train_s,
        times_train.copy(), events_train.copy())
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

    model = FusionModelV3(
        expr_dim=100, dysreg_dim=50, immune_dim=22, clinical_dim=5
    ).to(device)

    model, best_val_ci, best_epoch = train_fusion_model_v2(
        model, train_loader,
        val_expr, val_dysreg, val_immune, val_clinical,
        val_times, val_events,
        epochs=300, patience=30, lr=0.001
    )

    model.eval()
    with torch.no_grad():
        test_risk, test_attn = model(
            torch.FloatTensor(expr_test_s).to(device),
            torch.FloatTensor(dysreg_test_s).to(device),
            torch.FloatTensor(immune_test_s).to(device),
            torch.FloatTensor(clinical_test_s).to(device)
        )

    test_risk = test_risk.squeeze().cpu().numpy()
    test_attn = test_attn.cpu().numpy()

    ci_test = concordance_index_censored(
        events_test.astype(bool), times_test, test_risk)[0]

    fold_cindex.append(ci_test)
    fold_attn_weights.append(test_attn)

    print(f"{fold:<6} {best_epoch:<12} {ci_test:.4f}")

print("-" * 32)
print(f"\nFusion Model V3 C-index: {np.mean(fold_cindex):.3f} ± {np.std(fold_cindex):.3f}")
print(f"\nFull model comparison:")
print(f"  Cox Clinical:           0.700")
print(f"  Cox-Lasso Expression:   0.649")
print(f"  Cox-Lasso Dysreg:       0.559")
print(f"  Cox-Lasso Immune:       0.541")
print(f"  DeepSurv Expression:    0.569")
print(f"  Fusion V1 (3 streams):  0.610")
print(f"  Fusion V2 (4 streams):  0.632")
print(f"  Fusion V3 (simplified): {np.mean(fold_cindex):.3f}")

Running leakage-free 5-fold CV — FusionModelV3...
Gene selection inside each fold (top 100 expr, top 50 dysreg)
Fold   Best Epoch   Test C-index
--------------------------------
1      113          0.5994
2      103          0.6640
3      78           0.6543
4      98           0.7084
5      135          0.6391
--------------------------------

Fusion Model V3 C-index: 0.653 ± 0.035

Full model comparison:
  Cox Clinical:           0.700
  Cox-Lasso Expression:   0.649
  Cox-Lasso Dysreg:       0.559
  Cox-Lasso Immune:       0.541
  DeepSurv Expression:    0.569
  Fusion V1 (3 streams):  0.610
  Fusion V2 (4 streams):  0.632
  Fusion V3 (simplified): 0.653


In [49]:
!pip install lifelines -q
from lifelines import CoxPHFitter
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import VarianceThreshold

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_cindex       = []
fold_attn_weights = []

print("Running leakage-free 5-fold CV — FusionModelV3 + Cox gene selection...")
print("Gene selection inside each fold (top 30 expr, top 20 dysreg by Cox p-value)")
print(f"{'Fold':<6} {'Expr genes':<12} {'Dysreg genes':<14} {'Best Epoch':<12} {'Test C-index':<12}")
print("-" * 60)

for fold, (train_idx, test_idx) in enumerate(kf.split(expr, y['event']), 1):

    expr_train,     expr_test     = expr.iloc[train_idx],              expr.iloc[test_idx]
    dysreg_train,   dysreg_test   = dysreg.iloc[train_idx],            dysreg.iloc[test_idx]
    immune_train,   immune_test   = immune.iloc[train_idx],            immune.iloc[test_idx]
    clinical_train, clinical_test = clinical_features.iloc[train_idx], clinical_features.iloc[test_idx]
    y_train,        y_test        = y[train_idx],                      y[test_idx]

    times_train  = y_train['time'].copy()
    events_train = y_train['event'].copy()
    times_test   = y_test['time'].copy()
    events_test  = y_test['event'].copy()

    # ── Survival-guided gene selection (training patients only) ──────
    # Expression — top 30 by univariate Cox p-value
    cox_pvals_expr = {}
    for gene in expr_train.columns:
        try:
            df_tmp = pd.DataFrame({
                'T': times_train,
                'E': events_train,
                'gene': expr_train[gene].values
            })
            cph = CoxPHFitter()
            cph.fit(df_tmp, duration_col='T', event_col='E', show_progress=False)
            cox_pvals_expr[gene] = cph.summary['p'].values[0]
        except:
            cox_pvals_expr[gene] = 1.0
    top_expr_genes = pd.Series(cox_pvals_expr).nsmallest(30).index

    # Dysregulation — top 20 by univariate Cox p-value
    cox_pvals_dysreg = {}
    for gene in dysreg_train.columns:
        try:
            df_tmp = pd.DataFrame({
                'T': times_train,
                'E': events_train,
                'gene': dysreg_train[gene].values
            })
            cph = CoxPHFitter()
            cph.fit(df_tmp, duration_col='T', event_col='E', show_progress=False)
            cox_pvals_dysreg[gene] = cph.summary['p'].values[0]
        except:
            cox_pvals_dysreg[gene] = 1.0
    top_dysreg_genes = pd.Series(cox_pvals_dysreg).nsmallest(20).index
    # ────────────────────────────────────────────────────────────────

    expr_train_sel   = expr_train[top_expr_genes]
    expr_test_sel    = expr_test[top_expr_genes]
    dysreg_train_sel = dysreg_train[top_dysreg_genes]
    dysreg_test_sel  = dysreg_test[top_dysreg_genes]

    # Scale on training only
    scaler_expr     = StandardScaler()
    scaler_dysreg   = StandardScaler()
    scaler_immune   = StandardScaler()
    scaler_clinical = StandardScaler()

    expr_train_s     = scaler_expr.fit_transform(expr_train_sel)
    expr_test_s      = scaler_expr.transform(expr_test_sel)
    dysreg_train_s   = scaler_dysreg.fit_transform(dysreg_train_sel)
    dysreg_test_s    = scaler_dysreg.transform(dysreg_test_sel)
    immune_train_s   = scaler_immune.fit_transform(immune_train)
    immune_test_s    = scaler_immune.transform(immune_test)
    clinical_train_s = scaler_clinical.fit_transform(clinical_train)
    clinical_test_s  = scaler_clinical.transform(clinical_test)

    val_size     = int(0.2 * len(train_idx))
    val_expr     = torch.FloatTensor(expr_train_s[:val_size])
    val_dysreg   = torch.FloatTensor(dysreg_train_s[:val_size])
    val_immune   = torch.FloatTensor(immune_train_s[:val_size])
    val_clinical = torch.FloatTensor(clinical_train_s[:val_size])
    val_times    = times_train[:val_size].copy()
    val_events   = events_train[:val_size].copy()

    train_dataset = SurvivalDatasetV2(
        expr_train_s, dysreg_train_s, immune_train_s, clinical_train_s,
        times_train.copy(), events_train.copy())
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

    model = FusionModelV3(
        expr_dim=30, dysreg_dim=20, immune_dim=22, clinical_dim=5
    ).to(device)

    model, best_val_ci, best_epoch = train_fusion_model_v2(
        model, train_loader,
        val_expr, val_dysreg, val_immune, val_clinical,
        val_times, val_events,
        epochs=300, patience=30, lr=0.001
    )

    model.eval()
    with torch.no_grad():
        test_risk, test_attn = model(
            torch.FloatTensor(expr_test_s).to(device),
            torch.FloatTensor(dysreg_test_s).to(device),
            torch.FloatTensor(immune_test_s).to(device),
            torch.FloatTensor(clinical_test_s).to(device)
        )

    test_risk = test_risk.squeeze().cpu().numpy()
    test_attn = test_attn.cpu().numpy()

    ci_test = concordance_index_censored(
        events_test.astype(bool), times_test, test_risk)[0]

    fold_cindex.append(ci_test)
    fold_attn_weights.append(test_attn)

    print(f"{fold:<6} {len(top_expr_genes):<12} {len(top_dysreg_genes):<14} {best_epoch:<12} {ci_test:.4f}")

print("-" * 60)
print(f"\nFusion Model V3 + Cox Selection C-index: {np.mean(fold_cindex):.3f} ± {np.std(fold_cindex):.3f}")
print(f"\nFull model comparison:")
print(f"  Cox Clinical:                    0.700")
print(f"  Cox-Lasso Expression:            0.649")
print(f"  Fusion V3 (variance selection):  0.653")
print(f"  Fusion V3 (Cox selection):       {np.mean(fold_cindex):.3f}")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 13.9 MB/s eta 0:00:00
Running leakage-free 5-fold CV — FusionModelV3 + Cox gene selection...
Gene selection inside each fold (top 30 expr, top 20 dysreg by Cox p-value)
Fold   Expr genes   Dysreg genes   Best Epoch   Test C-index
------------------------------------------------------------
1      30           20             112          0.6188
2      30           20             106          0.6693
3      30           20             71           0.6145
4      30           20             154          0.6751
5      30           20             126          0.6955
------------------------------------------------------------

Fusion Model V3 + Cox Selection C-index: 0.655 ± 0.032

Full model comparison:
  Cox Clinical:                    0.700
  Cox-Lasso Expression:            0.649
  Fusion V3 (variance selectio

In [50]:
# Analyse attention weights across all folds
# Stream order: expression, dysregulation, immune, clinical
stream_names = ['Expression', 'Dysregulation', 'Immune', 'Clinical']

all_weights = np.concatenate(fold_attn_weights, axis=0)
mean_weights = all_weights.mean(axis=0)

print("Mean attention weights across all patients and folds:")
print("-" * 45)
for name, weight in zip(stream_names, mean_weights):
    bar = '█' * int(weight * 100)
    print(f"  {name:<15} {weight:.4f}  {bar}")

print("-" * 45)
print(f"  Total: {mean_weights.sum():.4f} (should be 1.0)")

Mean attention weights across all patients and folds:
---------------------------------------------
  Expression      0.2758  ███████████████████████████
  Dysregulation   0.1461  ██████████████
  Immune          0.3733  █████████████████████████████████████
  Clinical        0.2049  ████████████████████
---------------------------------------------
  Total: 1.0000 (should be 1.0)


In [51]:
import os
import shutil

# Create experiment folder
os.makedirs('experiments/v3_cox_selection', exist_ok=True)

# Save model
torch.save(model_final.state_dict(),
           'experiments/v3_cox_selection/fusion_model_v3_cox.pt')

# Save results
results_v3_cox = {
    "model": "FusionModelV3",
    "gene_selection": "univariate Cox p-value",
    "cv_strategy": "StratifiedKFold_5fold",
    "streams": ["expression", "dysregulation", "immune", "clinical"],
    "expr_genes": 30,
    "dysreg_genes": 20,
    "immune_features": 22,
    "clinical_features": 5,
    "cv_cindex_mean": 0.655,
    "cv_cindex_std": 0.032,
    "fold_cindices": [0.6188, 0.6693, 0.6145, 0.6751, 0.6955],
    "attention_weights": {
        "Expression": 0.2758,
        "Dysregulation": 0.1461,
        "Immune": 0.3733,
        "Clinical": 0.2049
    }
}

with open('experiments/v3_cox_selection/results.json', 'w') as f:
    json.dump(results_v3_cox, f, indent=2)

# Save scalers
with open('experiments/v3_cox_selection/scaler_expr.pkl', 'wb') as f:
    pickle.dump(scaler_expr, f)
with open('experiments/v3_cox_selection/scaler_dysreg.pkl', 'wb') as f:
    pickle.dump(scaler_dysreg, f)
with open('experiments/v3_cox_selection/scaler_immune.pkl', 'wb') as f:
    pickle.dump(scaler_immune, f)
with open('experiments/v3_cox_selection/scaler_clinical.pkl', 'wb') as f:
    pickle.dump(scaler_clinical, f)

print("Saved experiment: experiments/v3_cox_selection/")
print(f"  fusion_model_v3_cox.pt")
print(f"  results.json")
print(f"  scaler_expr.pkl")
print(f"  scaler_dysreg.pkl")
print(f"  scaler_immune.pkl")
print(f"  scaler_clinical.pkl")
print(f"\nCurrent best: 0.655 ± 0.032")

Saved experiment: experiments/v3_cox_selection/
  fusion_model_v3_cox.pt
  results.json
  scaler_expr.pkl
  scaler_dysreg.pkl
  scaler_immune.pkl
  scaler_clinical.pkl

Current best: 0.655 ± 0.032
